# Text Sentiment Classification: TF-IDF + Logistic Regression
Upload a CSV with one text column and one label column. Set the two column names in the first code cell.

## 1. Load data

In [ ]:
import re
import nltk
import pandas as pd
from google.colab import files
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

TEXT = 'text'
LABEL = 'label'

up = files.upload()
df = pd.read_csv(next(iter(up)))
df = df[[TEXT, LABEL]].dropna()
df[LABEL].value_counts()

## 2. Preprocess: lowercase, remove punctuation and stopwords, tokenize, lemmatize

In [ ]:
stop = set(stopwords.words('english'))
lem = WordNetLemmatizer()

def clean(t):
    t = re.sub(r'[^a-z\s]', '', t.lower())
    return ' '.join(lem.lemmatize(w) for w in t.split() if w not in stop)

df['clean'] = df[TEXT].astype(str).apply(clean)
df[[TEXT, 'clean']].head()

## 3. Split and build TF-IDF features

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(df['clean'], df[LABEL], test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=5000)
X_train = tfidf.fit_transform(X_tr)
X_test = tfidf.transform(X_te)
X_train.shape

## 4. Train and evaluate

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_tr)
pred = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_te, pred), 4))
print('F1 (weighted):', round(f1_score(y_te, pred, average='weighted'), 4))
print(classification_report(y_te, pred))

## 5. Top 10 words per class

In [ ]:
words = tfidf.get_feature_names_out()
top = {}

if len(model.classes_) == 2:
    order = model.coef_[0].argsort()
    top[model.classes_[0]] = words[order[:10]]
    top[model.classes_[1]] = words[order[-10:][::-1]]
else:
    for c, coef in zip(model.classes_, model.coef_):
        top[c] = words[coef.argsort()[-10:][::-1]]

pd.DataFrame(top)